# CHRONOS | Planet Nine candidate screening and posterior-sky analysis

**Kaggle Notebook edition.** Enable **Notebook options → Internet = On**. This notebook retrieves the public CaltechDATA reference population and (optionally) IRAS/AKARI catalogues through CDS VizieR. If you have real candidate astrometry, upload it as a Kaggle Dataset and attach it with **Add Input**.

This notebook uses the public **CaltechDATA Planet Nine reference population, Version 3.0** (100,000 synthetic orbital-element/physical-property samples; it incorporates ZTF, DES and Pan-STARRS constraints) and evaluates a published archival IRAS–AKARI candidate pair.

**This is not a claim that Planet Nine has been discovered.** The Caltech file is a synthetic reference population drawn from a particular dynamical/inference model, not astrometry of a detected planet. The 2025 IRAS–AKARI pair is two archival source positions separated by about 23 years; that is not enough information to determine a unique orbit. The notebook will not invent the missing orbital epoch or report a validated ephemeris unless a correctly defined orbital state or repeated astrometry is supplied.

The notebook downloads/checksums the public population and README, inspects the real schema, reproduces the published angular separation, optionally performs broad archival catalogue triage, and validates a supplied astrometry table. Any model-conditional sky propagation remains disabled unless epoch and element conventions are verified from the dataset documentation.

**Kaggle paths:** inputs are mounted under `/kaggle/input`; all outputs go under `/kaggle/working/chronos_planet9_work`. Save a notebook version to preserve outputs, then download the result ZIP from its Output/Files panel.

Sources: [CaltechDATA reference population V3](https://data.caltech.edu/records/8fjad-x7y61), [The Orbit of Planet Nine (Brown & Batygin 2021)](https://doi.org/10.3847/1538-3881/ac2056). Recheck current candidate status and literature before publishing conclusions.


In [ ]:
# Kaggle Internet must be ON for package installation and archive queries.
%pip -q install requests pandas numpy matplotlib scipy astropy astroquery


In [ ]:
from pathlib import Path
import os, json, hashlib, requests, re, zipfile, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, BarycentricTrueEcliptic, get_body_barycentric
from astropy.time import Time
import astropy.units as u

WORK=Path('/kaggle/working/chronos_planet9_work'); WORK.mkdir(parents=True, exist_ok=True)
DATA=WORK/'data'; DATA.mkdir(parents=True, exist_ok=True)
OUT=WORK/'results'; OUT.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT=Path('/kaggle/input')
print('Work directory:',WORK)
print('Kaggle input root exists:',KAGGLE_INPUT.exists())


## 1. Download the primary CaltechDATA V3 reference population

The official record lists `reference_population.csv` (about 13 MB) and `README.TXT`. The code uses the record's Invenio API to resolve the current download links, with a direct URL fallback. It records the downloaded byte size and checksums.

In [ ]:
RECORD_ID='8fjad-x7y61'
API_URL=f'https://data.caltech.edu/api/records/{RECORD_ID}'
RECORD_PAGE=f'https://data.caltech.edu/records/{RECORD_ID}'

# Offline path: attach reference_population.csv and README.TXT as a Kaggle Dataset (Add Input); no internet needed then.
attached=[p for p in KAGGLE_INPUT.rglob('reference_population.csv')] if KAGGLE_INPUT.exists() else []
try:
    r=requests.get(API_URL,timeout=30); r.raise_for_status(); record=r.json(); ONLINE=True
except Exception as exc:
    record={}; ONLINE=False; print('CaltechDATA not reachable (',type(exc).__name__,'); using attached files.')
entries=record.get('files',{}).get('entries',[])
print('Record title:',record.get('metadata',{}).get('title'))
print('File entries:',[e.get('key') for e in entries])

def resolve_file(name):
    for ent in entries:
        if ent.get('key')==name:
            links=ent.get('links',{})
            for k in ['content','self','download']:
                if links.get(k): return requests.compat.urljoin('https://data.caltech.edu',links[k])
    return f'https://data.caltech.edu/records/{RECORD_ID}/files/{name}?download=1'

def fetch(url,dest):
    rr=requests.get(url,timeout=240,stream=True,allow_redirects=True)
    rr.raise_for_status()
    tmp=Path(str(dest)+'.part')
    with open(tmp,'wb') as f:
        for chunk in rr.iter_content(1024*1024):
            if chunk: f.write(chunk)
    tmp.replace(dest)
    return dest

import shutil
if attached:
    csv_path=DATA/'reference_population.csv'; shutil.copy2(attached[0],csv_path)
    rm=list(attached[0].parent.glob('README*')); readme_path=DATA/'README.TXT'
    if rm: shutil.copy2(rm[0],readme_path)
    else: readme_path.write_text('README not attached; check https://data.caltech.edu/records/8fjad-x7y61')
    print('Using attached population:',attached[0])
elif ONLINE:
    csv_path=fetch(resolve_file('reference_population.csv'),DATA/'reference_population.csv')
    readme_path=fetch(resolve_file('README.TXT'),DATA/'README.TXT')
else:
    raise FileNotFoundError('No internet and no attached reference_population.csv. Attach it via Add Input or turn Internet on.')

def digest(p,algo='sha256'):
    h=hashlib.new(algo)
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

print('CSV bytes:',csv_path.stat().st_size,'SHA256:',digest(csv_path))
print('README bytes:',readme_path.stat().st_size)
print('\n--- Dataset README ---\n')
print(readme_path.read_text(errors='replace'))

In [ ]:
# Load without assuming column names or units.
df=pd.read_csv(csv_path)
print('Rows:',len(df),'Columns:',len(df.columns))
print('Columns:',df.columns.tolist())
display(df.head())
print('\nNumeric summary:')
display(df.describe(include='all').T)

## 2. Map orbital-element columns only after inspecting the real schema

The V3 record page describes 100,000 synthetic orbital elements, but this notebook deliberately does not hard-code unverified CSV column names or an epoch. Inspect the `README.TXT` and the printed schema above. Edit the mapping cell only when the units and coordinate convention are explicitly established.

In [ ]:
# Auto-detect common column labels, then VERIFY every mapping against README.TXT.
# If an alias is not recognized, set the value manually to an exact column name.
ALIASES = {
    'a_au':['a','a_au','sma','semimajor_axis','semi_major_axis'],
    'e':['e','ecc','eccentricity'],
    'i_deg':['i','inc','inclination','i_deg'],
    'Omega_deg':['Omega','Omega_deg','Omega_node','longitude_of_ascending_node'],
    'omega_deg':['omega','omega_deg','argperi','arg_peri','argument_of_perihelion'],
    'M_deg':['M','mean_anomaly','meananom','M_deg','mean_anomaly_deg'],
}
def resolve_exact_then_casefold(names):
    for name in names:
        if name in df.columns: return name
    lower_map={}
    for col in df.columns: lower_map.setdefault(str(col).strip().lower(),[]).append(col)
    hits=[]
    for name in names:
        hits.extend(lower_map.get(name.lower(),[]))
    hits=list(dict.fromkeys(hits))
    return hits[0] if len(hits)==1 else None
COLUMN_MAP={key:resolve_exact_then_casefold(names) for key,names in ALIASES.items()}
COLUMN_MAP['epoch']=None
print('Automatically suggested mapping (not yet endorsed):',COLUMN_MAP)
print('Verify field meanings and units against README.TXT before continuing.')

# Override any incorrect or None value using an exact CSV header string, e.g. COLUMN_MAP['a_au']='a'.
# Leave a value as None if the field/units cannot be confirmed.
# COLUMN_MAP['a_au'] = None

ORBITAL_EPOCH_TDB = None  # Must be documented by the data README / original methods.
ELEMENT_FRAME = None      # Must be documented; implementation below supports heliocentric ecliptic J2000 only.

required=['a_au','e','i_deg','Omega_deg','omega_deg','M_deg']
missing=[k for k in required if COLUMN_MAP.get(k) is None]
actual_missing=[COLUMN_MAP[k] for k in required if COLUMN_MAP.get(k) is not None and COLUMN_MAP[k] not in df.columns]
if actual_missing: raise KeyError(f'Mapped columns not in CSV: {actual_missing}')
if missing:
    print('POSTERIOR PROPAGATION DISABLED. Unmapped fields:',missing)
else:
    print('All six orbital fields have candidate mappings. Confirm units, reference epoch and element frame before enabling propagation.')


In [ ]:
# Exploratory posterior distributions can be plotted only for columns with confirmed mappings.
if not missing:
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,key,label in zip(axes.ravel(),required,['a (AU)','e','i (deg)','Omega (deg)','omega (deg)','M (deg)']):
        vals=pd.to_numeric(df[COLUMN_MAP[key]],errors='coerce').dropna()
        ax.hist(vals,bins=60); ax.set_title(label); ax.set_ylabel('sample count')
    plt.tight_layout(); plt.savefig(OUT/'posterior_parameter_distributions.png',dpi=160); plt.show()
else:
    print('No posterior plots created because the column map is not yet supported by verified metadata.')

## 3. Reproduce a published IRAS–AKARI two-position candidate pair

The published pair is a useful candidate-triage example, not a discovered planet. The code independently recomputes the angular separation. The observation baseline in the paper is approximately 23 years; do not infer an orbit from just these two positions.

In [ ]:
iras=SkyCoord(ra=35.74075*u.deg,dec=-48.5125*u.deg,frame='icrs')
akari=SkyCoord(ra=35.18379*u.deg,dec=-49.2135*u.deg,frame='icrs')
sep=iras.separation(akari)
baseline_years=23.0  # approximate published interval; replace only with exact observation dates from source tables
rate=sep.to(u.arcsec).value/baseline_years
candidate_pair={
 'iras_ra_deg':float(iras.ra.deg),'iras_dec_deg':float(iras.dec.deg),
 'akari_ra_deg':float(akari.ra.deg),'akari_dec_deg':float(akari.dec.deg),
 'separation_arcmin':float(sep.to(u.arcmin).value),
 'assumed_baseline_years':baseline_years,
 'mean_angular_rate_arcsec_per_year':float(rate),
 'qualification':'two archive positions only; not sufficient for a unique orbit or validated ephemeris'
}
(OUT/'published_candidate_pair_check.json').write_text(json.dumps(candidate_pair,indent=2))
print(json.dumps(candidate_pair,indent=2))

## 3B. Optional all-sky IRAS–AKARI candidate triage

This optional step queries the IRAS Point Source Catalog (`II/125/main`) and AKARI/FIS Bright Source Catalog (`II/298/fis`) through VizieR, then searches for widely separated positional pairs in the approximate long time-baseline window. The nominal catalog sizes are 245,889 and 427,071 sources. The known 47.46-arcmin published pair is a useful reference check.

**This is deliberately only a broad triage list.** A large angular separation can be a chance association, confusion/blending, extended emission, or a catalog-position issue. The survey catalogue epochs are not identical point-epochs for every source. Every candidate needs original survey-image inspection, source-quality review, cross-checks against other catalogues, and independent follow-up before it can be described as a moving object. The output is not an orbit or discovery.

VizieR table definitions: [IRAS PSC II/125](https://vizier.cds.unistra.fr/cgi-bin/VizieR-3?-source=II%2F125) and [AKARI/FIS II/298](https://vizier.cds.unistra.fr/cgi-bin/VizieR-3?-source=II%2F298).

In [ ]:
# Optional full-catalog crossmatch. It can take several minutes and uses internet/data transfer.
def _online(url='https://vizier.cds.unistra.fr'):
    try: return requests.get(url,timeout=10).ok
    except Exception: return False
RUN_ARCHIVE_CROSSMATCH = _online()  # runs only when VizieR is reachable; offline runs skip it and say so
MAX_PAIR_RESULTS = 500000
TOP_AKARI_MATCHES_PER_IRAS = 3
CROSSMATCH_BATCH_ROWS = 1500
MIN_SEPARATION_DEG = 0.15   # 9 arcmin; suppress ordinary stationary matches
MAX_SEPARATION_DEG = 0.90   # includes the published ~47.5 arcmin example
ECLIPTIC_LAT_MAX_DEG = 40.0 # broad model-motivated screening band, not a proof of location

archive_crossmatch_summary={'enabled':bool(RUN_ARCHIVE_CROSSMATCH)}
if RUN_ARCHIVE_CROSSMATCH:
    from astroquery.vizier import Vizier
    from tqdm.auto import tqdm
    from astropy.coordinates import FK4, search_around_sky, BarycentricTrueEcliptic
    v=Vizier(columns=['**'],row_limit=-1)
    print('Downloading IRAS PSC II/125/main ...')
    itables=v.get_catalogs('II/125/main')
    print('Downloading AKARI/FIS II/298/fis ...')
    atables=v.get_catalogs('II/298/fis')
    if len(itables)==0 or len(atables)==0: raise RuntimeError('VizieR returned no table(s). Retry later or inspect VizieR service status.')
    ir=itables[0]; ak=atables[0]
    print('IRAS:',len(ir),ir.colnames)
    print('AKARI:',len(ak),ak.colnames)
    # Save exact public catalog snapshots for reproducibility.
    ir.write(DATA/'iras_psc_vizier_snapshot.ecsv',format='ascii.ecsv',overwrite=True)
    ak.write(DATA/'akari_fis_vizier_snapshot.ecsv',format='ascii.ecsv',overwrite=True)
    (OUT/'archive_catalog_query.json').write_text(json.dumps({'IRAS_catalog':'II/125/main','AKARI_catalog':'II/298/fis',
       'IRAS_rows':len(ir),'AKARI_rows':len(ak),'IRAS_columns':ir.colnames,'AKARI_columns':ak.colnames,
       'snapshot_format':'ECSV','catalog_source':'CDS VizieR'},indent=2))

    def first_column(tab,names):
        for n in names:
            if n in tab.colnames: return n
        return None
    # IRAS J2000 positions if present, otherwise transform published B1950 coordinates.
    ira=first_column(ir,['RAJ2000','_RAJ2000','ra','RA'])
    ide=first_column(ir,['DEJ2000','_DEJ2000','dec','DEC'])
    if ira and ide:
        ir_coords=SkyCoord(ra=np.asarray(ir[ira],float)*u.deg,dec=np.asarray(ir[ide],float)*u.deg,frame='icrs')
        ir_position_method=f'catalog fields {ira}/{ide}'
    elif 'RA1950' in ir.colnames and 'DE1950' in ir.colnames:
        ra_raw=np.asarray(ir['RA1950'])
        dec_raw=np.asarray(ir['DE1950'])
        if ra_raw.dtype.kind in 'USO':
            ir_fk4=SkyCoord(ra=ra_raw.astype(str),dec=dec_raw.astype(str),unit=(u.hourangle,u.deg),
                            frame=FK4(equinox=Time(1950.0,format='byear'),obstime=Time(1983.5,format='jyear')))
        else:
            # Catalog documentation labels RA1950 as sexagesimal hours. If VizieR returns numerics, interpret as hours.
            ir_fk4=SkyCoord(ra=ra_raw.astype(float)*u.hourangle,dec=dec_raw.astype(float)*u.deg,
                            frame=FK4(equinox=Time(1950.0,format='byear'),obstime=Time(1983.5,format='jyear')))
        ir_coords=ir_fk4.icrs
        ir_position_method='RA1950/DE1950 transformed from FK4 B1950; epoch metadata 1983.5'
    else:
        raise KeyError('Could not locate documented IRAS coordinate columns; inspect printed schema before editing.')

    ara=first_column(ak,['RAdeg','RAfis','RAJ2000','RA'])
    ade=first_column(ak,['DEdeg','DEfis','DEJ2000','DEC','DE'])
    if not ara or not ade: raise KeyError('Could not locate AKARI RA/Dec columns; inspect printed schema before editing.')
    ak_coords=SkyCoord(ra=np.asarray(ak[ara],float)*u.deg,dec=np.asarray(ak[ade],float)*u.deg,frame='icrs')

    def quality_mask(tab,candidates):
        qcol=first_column(tab,candidates)
        if qcol is None: return np.ones(len(tab),bool),None
        q=np.asarray(tab[qcol],float)
        return np.isfinite(q)&(q>=2),qcol
    ir_good,ir_qcol=quality_mask(ir,['q_Fnu_60','FQUAL_60','fqual_60','q60'])
    ak_good,ak_qcol=quality_mask(ak,['q_S90','q_S65'])
    # broad ecliptic restriction applied before pair search
    ir_ecl=ir_coords.transform_to(BarycentricTrueEcliptic(equinox=Time(2000.0,format='jyear'))).lat.deg
    ak_ecl=ak_coords.transform_to(BarycentricTrueEcliptic(equinox=Time(2000.0,format='jyear'))).lat.deg
    ir_keep=ir_good & (np.abs(ir_ecl)<=ECLIPTIC_LAT_MAX_DEG)
    ak_keep=ak_good & (np.abs(ak_ecl)<=ECLIPTIC_LAT_MAX_DEG)
    ir_idx_all=np.where(ir_keep)[0]; ak_idx_all=np.where(ak_keep)[0]
    ak_sky=ak_coords[ak_idx_all]
    print('Coordinate interpretation:',ir_position_method)
    print('Quality filter fields:',ir_qcol,ak_qcol)
    print('Sources after quality/ecliptic filter:',len(ir_idx_all),len(ak_idx_all))

    ir_name=first_column(ir,['IRAS','Name','pscname'])
    ak_name=first_column(ak,['objName','objID'])
    candidates=[]
    processed_iras=0
    scan_truncated=False
    for start in tqdm(range(0,len(ir_idx_all),CROSSMATCH_BATCH_ROWS),desc='IRAS x AKARI crossmatch'):
        ids=ir_idx_all[start:start+CROSSMATCH_BATCH_ROWS]
        i1,i2,sep2d,_=search_around_sky(ir_coords[ids],ak_sky,MAX_SEPARATION_DEG*u.deg)
        sepdeg=sep2d.deg
        keep=(sepdeg>=MIN_SEPARATION_DEG)&(sepdeg<=MAX_SEPARATION_DEG)
        # Avoid dumping millions of random angular pairs: retain only the nearest 3 AKARI
        # associations in the candidate-motion annulus for each IRAS source.
        local=[]
        for local_ir in np.unique(i1[keep]):
            loc=np.where(keep & (i1==local_ir))[0]
            loc=loc[np.argsort(sepdeg[loc])[:TOP_AKARI_MATCHES_PER_IRAS]]
            for l in loc:
                ii=int(ids[int(i1[l])]); jj=int(ak_idx_all[int(i2[l])])
                local.append({'iras_row':ii,'akari_row':jj,'nearest_rank_for_iras':0,
                              'iras_name':str(ir[ir_name][ii]) if ir_name else str(ii),
                              'akari_name':str(ak[ak_name][jj]) if ak_name else str(jj),
                              'separation_deg':float(sepdeg[l]),'separation_arcmin':float(sepdeg[l]*60),
                              'nominal_motion_arcsec_yr_23y':float(sepdeg[l]*3600/23.0),
                              'ecliptic_lat_iras_deg':float(ir_ecl[ii]),'iras_quality_column':ir_qcol,'akari_quality_column':ak_qcol})
        # Recompute stable within-source ranks explicitly for the selected pairs.
        if local:
            local_df=pd.DataFrame(local)
            local_df['nearest_rank_for_iras']=local_df.groupby('iras_row')['separation_deg'].rank(method='first').astype(int)
            candidates.extend(local_df.to_dict('records'))
        processed_iras += len(ids)
        if len(candidates)>=MAX_PAIR_RESULTS:
            print('Reached result cap; remaining IRAS sources were not processed.')
            scan_truncated=True
            break
    cand=pd.DataFrame(candidates).sort_values(['separation_arcmin','iras_row']) if candidates else pd.DataFrame()
    archive_crossmatch_summary={'enabled':True,'IRAS_rows_total':len(ir),'AKARI_rows_total':len(ak),
       'IRAS_sources_after_quality_and_ecliptic_filter':len(ir_idx_all),'IRAS_sources_processed':processed_iras,
       'candidate_pairs_written':len(cand),'truncated_by_result_cap':scan_truncated,
       'minimum_separation_deg':MIN_SEPARATION_DEG,'maximum_separation_deg':MAX_SEPARATION_DEG,
       'top_AKARI_matches_per_IRAS':TOP_AKARI_MATCHES_PER_IRAS,
       'interpretation':'triage associations only; not discoveries; many chance alignments are expected'}
    (OUT/'archive_crossmatch_summary.json').write_text(json.dumps(archive_crossmatch_summary,indent=2))
    cand.to_csv(OUT/'iras_akari_wide_separation_triage.csv',index=False)
    print('Triage pair count:',len(cand))
    if len(cand):
        near=cand.iloc[(cand.separation_arcmin-47.4586).abs().argsort()[:10]]
        print('Pairs nearest the published 47.46 arcmin example (not necessarily the same sources):')
        display(near)
    print('This output is a candidate-association list only. Do not call any row a discovery without image confirmation and independent follow-up.')
else:
    print('IRAS–AKARI candidate crossmatch skipped by setting. Set RUN_ARCHIVE_CROSSMATCH=True to execute in Kaggle.')


## 4. Optional model-conditional sky ensemble (strictly gated)

This propagation is enabled only if the following are all verified: (a) all six orbital columns are correctly mapped, (b) their units are confirmed, (c) the exact sample epoch is documented, and (d) the element frame is known. The resulting plot is a **posterior/model-conditional sky-location distribution**, not an observed-object ephemeris.

In [ ]:
CAN_PROPAGATE=(not missing and ORBITAL_EPOCH_TDB is not None and ELEMENT_FRAME is not None)
print('CAN_PROPAGATE:',CAN_PROPAGATE)
if not CAN_PROPAGATE:
    print('Skipped by design: one or more mandatory orbital conventions / epoch are not verified.')
else:
    if ELEMENT_FRAME.lower() not in ['heliocentric ecliptic j2000','ecliptic j2000 heliocentric']:
        raise ValueError('This notebook only implements the documented heliocentric ecliptic J2000 convention; do not silently transform another frame.')
    from astropy.coordinates import ICRS
    # Kepler solver for elliptic orbits.
    def solve_kepler(M,e,niter=20):
        M=np.mod(M,2*np.pi); E=M.copy()
        for _ in range(niter):
            d=(E-e*np.sin(E)-M)/(1-e*np.cos(E))
            E-=d
            if np.nanmax(np.abs(d))<1e-12: break
        return E
    # Subsample for plotting and compute heliocentric positions from osculating elements.
    sample=df.sample(min(20000,len(df)),random_state=42)
    a=pd.to_numeric(sample[COLUMN_MAP['a_au']],errors='coerce').to_numpy(float)
    e=pd.to_numeric(sample[COLUMN_MAP['e']],errors='coerce').to_numpy(float)
    inc=np.deg2rad(pd.to_numeric(sample[COLUMN_MAP['i_deg']],errors='coerce').to_numpy(float))
    Om=np.deg2rad(pd.to_numeric(sample[COLUMN_MAP['Omega_deg']],errors='coerce').to_numpy(float))
    om=np.deg2rad(pd.to_numeric(sample[COLUMN_MAP['omega_deg']],errors='coerce').to_numpy(float))
    M0=np.deg2rad(pd.to_numeric(sample[COLUMN_MAP['M_deg']],errors='coerce').to_numpy(float))
    good=np.isfinite(a+e+inc+Om+om+M0)&(a>0)&(e>=0)&(e<1)
    a,e,inc,Om,om,M0=[x[good] for x in [a,e,inc,Om,om,M0]]
    epoch=Time(ORBITAL_EPOCH_TDB,scale='tdb')
    # Dates: near-term forecast sampling; adjust horizon without altering posterior.
    dates=Time(['2026-11-14','2027-01-01','2027-07-01','2028-01-01'],scale='tdb')
    # Gaussian gravitational constant in AU^(3/2)/day for solar mass; two-body Kepler approximation.
    mu=0.0002959122082855911 # AU^3/day^2
    period_days=2*np.pi*np.sqrt(a**3/mu)
    eps=23.439291111*np.pi/180
    R_ecl_to_eq=np.array([[1,0,0],[0,np.cos(eps),-np.sin(eps)],[0,np.sin(eps),np.cos(eps)]])
    for date in dates:
        dt=(date-epoch).to_value(u.day)
        M=M0+2*np.pi*dt/period_days
        E=solve_kepler(M,e)
        xv=a*(np.cos(E)-e); yv=a*np.sqrt(1-e**2)*np.sin(E)
        # perifocal -> ecliptic J2000 inertial rotation Rz(Omega) Rx(i) Rz(omega)
        cw=np.cos(om); sw=np.sin(om); cO=np.cos(Om); sO=np.sin(Om); ci=np.cos(inc); si=np.sin(inc)
        x=(cO*cw-sO*sw*ci)*xv+(-cO*sw-sO*cw*ci)*yv
        y=(sO*cw+cO*sw*ci)*xv+(-sO*sw+cO*cw*ci)*yv
        z=(sw*si)*xv+(cw*si)*yv
        eq=R_ecl_to_eq@np.vstack([x,y,z])
        t=SkyCoord(x=eq[0]*u.au,y=eq[1]*u.au,z=eq[2]*u.au,representation_type='cartesian',frame=ICRS())
        sun=get_body_barycentric('sun',date).xyz.to_value(u.au)
        earth=get_body_barycentric('earth',date).xyz.to_value(u.au)
        helio=np.vstack([t.cartesian.x.to_value(u.au),t.cartesian.y.to_value(u.au),t.cartesian.z.to_value(u.au)])
        geo=helio+sun[:,None]-earth[:,None]
        c=SkyCoord(x=geo[0]*u.au,y=geo[1]*u.au,z=geo[2]*u.au,representation_type='cartesian',frame=ICRS())
        print(date.isot,'valid ensemble N=',len(c),'RA median=',np.median(c.ra.deg),'Dec median=',np.median(c.dec.deg))
        fig=plt.figure(figsize=(7,5)); ax=fig.add_subplot(111)
        ax.scatter(c.ra.wrap_at(180*u.deg).deg,c.dec.deg,s=1,alpha=.12)
        ax.set_xlabel('RA (deg, wrapped)'); ax.set_ylabel('Dec (deg)'); ax.set_title('Model-conditional Planet Nine sky ensemble '+date.isot)
        ax.invert_xaxis(); plt.tight_layout(); plt.savefig(OUT/f'planet9_model_sky_{date.datetime.date()}.png',dpi=160); plt.show()

## 5. Validate candidate astrometry supplied as a Kaggle Dataset

This template is only a data-format gate. Three distinct epochs are a bare minimum for inspecting motion; they do not guarantee a unique or accurate orbit. A defensible orbit requires well-calibrated detections across a sufficiently long arc, observer locations, covariance/uncertainties, image-level verification, and independent follow-up.

Upload `planet9_candidate_astrometry.csv` as a Kaggle Dataset and attach it with **Add Input**. Required fields:

`date_utc,ra_deg,dec_deg,sigma_ra_arcsec,sigma_dec_arcsec,observatory_code`

Also include original image/cutout identifiers, filter/band, RA-Dec covariance if available, and astrometric reference-solution metadata. If more than one matching CSV is attached, set `ASTROMETRY_CSV_OVERRIDE` in the next cell.


In [ ]:
# Optional: locate candidate astrometry attached through Kaggle Add Input.
ASTROMETRY_CSV_OVERRIDE=None  # e.g. '/kaggle/input/my-candidate-data/planet9_candidate_astrometry.csv'
ASTROMETRY_CSV=None
if ASTROMETRY_CSV_OVERRIDE:
    p=Path(ASTROMETRY_CSV_OVERRIDE)
    if not p.exists(): raise FileNotFoundError(p)
    ASTROMETRY_CSV=str(p)
elif KAGGLE_INPUT.exists():
    candidates=[p for p in KAGGLE_INPUT.rglob('*.csv') if p.name.lower() in ('planet9_candidate_astrometry.csv','candidate_astrometry.csv')]
    if len(candidates)==1: ASTROMETRY_CSV=str(candidates[0])
    elif len(candidates)>1:
        print('Multiple candidate astrometry CSVs found; set ASTROMETRY_CSV_OVERRIDE explicitly:')
        for p in candidates: print(' ',p)
    else: print('No candidate astrometry CSV attached. This is expected if you have no verified moving-object observations.')
print('ASTROMETRY_CSV:',ASTROMETRY_CSV)


In [ ]:
# After uploading, set ASTROMETRY_CSV to the exact file path.
if ASTROMETRY_CSV is not None:
    obs=pd.read_csv(ASTROMETRY_CSV)
    required_cols=['date_utc','ra_deg','dec_deg','sigma_ra_arcsec','sigma_dec_arcsec','observatory_code']
    missing_cols=[c for c in required_cols if c not in obs]
    if missing_cols: raise ValueError('Candidate astrometry missing: '+str(missing_cols))
    obs['date_utc']=pd.to_datetime(obs.date_utc,utc=True,errors='raise')
    obs=obs.sort_values('date_utc')
    unique_epochs=obs.date_utc.nunique()
    sc=SkyCoord(ra=obs.ra_deg.to_numpy()*u.deg,dec=obs.dec_deg.to_numpy()*u.deg,frame='icrs')
    print('Measurements:',len(obs),'distinct epochs:',unique_epochs)
    print('Time span:',(obs.date_utc.max()-obs.date_utc.min()).total_seconds()/86400,'days')
    print('Max pairwise sky separation (arcsec):',float(np.max(sc.separation(sc[0]).arcsec)))
    if unique_epochs<3:
        raise RuntimeError('STOP: fewer than 3 distinct astrometric epochs. This is insufficient even for the preliminary motion gate.')
    print('Preliminary epoch-count gate passed only. This notebook does NOT fit an orbit or produce an ephemeris; review astrometry, baseline, covariance and observer geometry before orbit determination.')
else:
    print('No candidate astrometry uploaded. That is expected: the published IRAS–AKARI two-point pair is not enough to fit a unique orbit.')

## 6. Scientific stop rules and output

The all-sky IRAS–AKARI match is a candidate-triage search only. The catalog epochs, coordinate systems, source quality, blends, scan artifacts, source morphology and counterpart ambiguity must be checked against the original survey images. Any plausible candidate needs independent repeat detections and follow-up before it can be called a trans-Neptunian object or Planet Nine.

The synthetic CaltechDATA orbital population is a model prior/ensemble. It is not a list of observed objects. This notebook does not claim to discover a planet, fit an observed orbit, or publish validated ephemerides. It exports candidate pairs and quality-control information for further verification.


In [ ]:
summary={
 'caltechdata_record':RECORD_ID,
 'dataset_rows':int(len(df)),
 'dataset_columns':df.columns.tolist(),
 'csv_bytes':csv_path.stat().st_size,
 'csv_sha256':digest(csv_path),
 'published_candidate_pair':candidate_pair,
 'column_map':COLUMN_MAP,
 'orbital_epoch_tdb':ORBITAL_EPOCH_TDB,
 'element_frame':ELEMENT_FRAME,
 'model_sky_propagation_enabled':bool(CAN_PROPAGATE),
 'archive_crossmatch':archive_crossmatch_summary,
 'validated_planet_nine_ephemeris_produced':False,
 'reason':'No confirmed Planet Nine object and no independently validated observed orbit/astrometry are supplied. The reference population is a synthetic model ensemble.'
}
(OUT/'planet9_analysis_summary.json').write_text(json.dumps(summary,indent=2))
ZIP=WORK/'planet9_candidate_assessment_results.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for source_dir in [OUT,DATA]:
        for f in source_dir.rglob('*'):
            if f.is_file() and not f.name.endswith('.part'): z.write(f,f.relative_to(WORK))
print('Result package:',ZIP,'bytes:',ZIP.stat().st_size)
print('Individual CSV/JSON/PNG results:',OUT)
print('Kaggle: save a notebook version; download the ZIP or individual files from its Output/Files panel.')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(ZIP)))
except Exception:
    pass